# 1. Hybrid NRMS: History Self-Attention + Tabular MLP

Replaces the dot-product click predictor of pure NRMS with an MLP that also sees the
causal tabular features.

- **News encoder**: frozen 64-dimensional text embeddings shared by candidates and history
- **User encoder**:
    - **Self-attention block**: two-head self-attention over the clicked articles, followed by
      a residual connection, layer normalisation and a position-wise feed-forward network
    - **Additive attention**: pools the contextualised history into one user vector; users
      with no history get a learned default vector
- **MLP scorer**: concatenates tabular features, the candidate vector and the user vector

## 1.1 Load Packages

In [ ]:
import json
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = sorted((RUN / f"ranking_features_{split}").glob("*.parquet"))[0]
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

In [ ]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

EXPERIMENT_NAME = "hybrid_nrms_mlp_v1"
TRAIN_ROWS = 300_000
HISTORY_CAP = 50
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20
EPOCHS = 5
BATCH_SIZE = 2048
OUTPUT_DIR = ARTIFACTS / EXPERIMENT_NAME

In [ ]:
# The frozen feature schema lists the causal tabular inputs shared by every reranker
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [name for name in schema.get("categorical_columns", []) if name in FEATURES]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
print(f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")

# 3. Prepare Training Data

## 3.1 Sample Labelled Training Candidates

`sample_train_rows` draws a reproducible, uniform sample of training candidates. The
number of rows taken from each partition follows a hypergeometric draw, so the whole
training split never has to be loaded at once.

In [ ]:
def read_labelled_part(part, columns):
    """Join one training feature partition to its click labels."""
    features = pq.read_table(part, columns=["impression_uid", "news_code", *columns]).to_pandas()
    labels = pq.read_table(
        RUN / "labels_train" / part.name,
        columns=["impression_uid", "news_code", "clicked"],
    ).to_pandas()
    return features.merge(labels, on=["impression_uid", "news_code"], validate="one_to_one")


def sample_train_rows(limit, columns):
    """Sample `limit` labelled training candidates uniformly across all partitions."""
    parts = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    sizes = [pq.ParquetFile(part).metadata.num_rows for part in parts]
    rng = np.random.default_rng(SEED)
    remaining_population = sum(sizes)
    remaining_take = min(limit, remaining_population)

    samples = []
    for part, size in zip(parts, sizes):
        take = int(rng.hypergeometric(size, remaining_population - size, remaining_take))
        if take:
            frame = read_labelled_part(part, columns)
            samples.append(frame.iloc[rng.choice(size, take, replace=False)])
        remaining_population -= size
        remaining_take -= take
    return pd.concat(samples, ignore_index=True)

## 3.2 Article Vectors and Causal Click Histories

Every article is represented by its frozen 64-dimensional text embedding. A user's
history is the list of articles they clicked **strictly before** the impression, saved
by feature engineering; at most the newest `HISTORY_CAP` clicks are used.

In [ ]:
# Map each news_code to its row in the frozen 64-dimensional text embedding matrix.
# Row 0 is reserved for padding.
article_index = pl.read_parquet(REPRESENTATIONS / "article_index.parquet", columns=["news_code", "article_row"])
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]
print(f"Article embeddings: {text_vectors.shape[0]:,} rows x {EMBEDDING_DIM} dimensions")

In [ ]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = (
        pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet")
        .select("impression_uid", "full_history_rows")
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

In [ ]:
def encode_pairs(frame, profiles):
    """Return the padded click history and the candidate article row for every candidate.

    Histories keep the newest HISTORY_CAP clicks and are left-padded with row 0.
    """
    histories = np.zeros((len(frame), HISTORY_CAP), dtype=np.int64)
    candidates = np.empty(len(frame), dtype=np.int64)
    for index, (uid, news_code) in enumerate(zip(frame["impression_uid"], frame["news_code"])):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
        candidates[index] = news_to_row[news_code]
    return histories, candidates

## 3.3 Fit Feature Preprocessing

Fitted on the training sample only.

In [ ]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ])

In [ ]:
train = sample_train_rows(TRAIN_ROWS, columns=FEATURES)
preprocessor = build_preprocessor()
train_tabular = preprocessor.fit_transform(train[FEATURES]).astype(np.float32, copy=False)
train_profiles = load_profiles("train", train["impression_uid"].unique())
train_history, train_candidate = encode_pairs(train, train_profiles)
train_labels = train["clicked"].to_numpy(np.float32)
print(f"Training rows: {len(train):,}; tabular width: {train_tabular.shape[1]}; click rate: {train_labels.mean():.3%}")

# 4. Model

## 4.1 Additive Attention

Scores each contextualised clicked article with a learned query vector and returns their
weighted sum as the user vector. Padding receives zero weight.

In [ ]:
class AdditiveAttention(nn.Module):
    """Pool a sequence of history states into one user vector with learned relevance weights."""

    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        # states: [batch, history, dim]; padding_mask: [batch, history], True = padding
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False                       # softmax needs one open slot

        scores = torch.tanh(self.proj(states)) @ self.query    # [batch, history]
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)     # [batch, dim]

## 4.2 `NRMSUserEncoder`

```text
Clicked articles → multi-head self-attention → residual + LayerNorm
                 → feed-forward (4× width) → residual + LayerNorm
                 → additive attention → user vector
```

In [ ]:
class NRMSUserEncoder(nn.Module):
    """Self-attention over causal clicks followed by additive-attention pooling."""

    def __init__(self, articles, dim, heads, dropout):
        super().__init__()
        self.articles = articles
        self.self_attention = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dim, dim),
        )
        self.norm2 = nn.LayerNorm(dim)
        self.pool = AdditiveAttention(dim)
        self.default_user = nn.Parameter(torch.zeros(dim))   # used when there is no history

    def forward(self, history_rows):
        history = self.articles(history_rows)               # [batch, history, dim]
        padding_mask = history_rows.eq(0)
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False

        attended, _ = self.self_attention(history, history, history, key_padding_mask=safe_mask, need_weights=False)
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)              # [batch, dim]
        return torch.where(all_padded.unsqueeze(1), self.default_user.unsqueeze(0), user)

## 4.3 `HybridNRMSReranker`

```text
[tabular features ; candidate vector ; user vector] → Dense 256 → Dense 64 → logit
```

In [ ]:
class HybridNRMSReranker(nn.Module):
    """Article embeddings → NRMS user encoder → tabular MLP scorer."""

    def __init__(self, vectors, tabular_width):
        super().__init__()
        self.articles = nn.Embedding.from_pretrained(
            torch.from_numpy(np.array(vectors, dtype=np.float32, copy=True)),
            freeze=True,
            padding_idx=0,
        )
        self.user_encoder = NRMSUserEncoder(self.articles, EMBEDDING_DIM, ATTENTION_HEADS, ATTENTION_DROPOUT)

        layers, width = [], tabular_width + 2 * EMBEDDING_DIM
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, history_rows, candidate_rows):
        candidate = self.articles(candidate_rows)
        user = self.user_encoder(history_rows)
        return self.scorer(torch.cat([tabular, candidate, user], dim=1)).squeeze(-1)

# 5. Training

Weighted binary cross-entropy with a training-derived positive weight.

In [ ]:
torch.manual_seed(SEED)
model = HybridNRMSReranker(text_vectors, train_tabular.shape[1]).to(DEVICE)
click_rate = train_labels.mean()
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - click_rate) / click_rate], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loader = DataLoader(
    TensorDataset(
        torch.from_numpy(train_tabular),
        torch.from_numpy(train_history),
        torch.from_numpy(train_candidate),
        torch.from_numpy(train_labels),
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for tabular, histories, candidates, labels in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(tabular.to(DEVICE), histories.to(DEVICE), candidates.to(DEVICE)), labels.to(DEVICE))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f"Epoch {epoch + 1}/{EPOCHS}: train BCE={total_loss / len(train_labels):.5f}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save({"state_dict": model.state_dict(), "features": FEATURES, "history_cap": HISTORY_CAP, "seed": SEED},
           OUTPUT_DIR / "model.pt")
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

In [ ]:
def predict_batches(model, *inputs):
    """Score aligned NumPy inputs in mini-batches and return click probabilities."""
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(inputs[0]), BATCH_SIZE):
            batch = [torch.from_numpy(array[start:start + BATCH_SIZE]).to(DEVICE) for array in inputs]
            scores.append(torch.sigmoid(model(*batch)).cpu().numpy())
    return np.concatenate(scores)

In [ ]:
def hybrid_predictor(profiles):
    """Wrap the trained model and a split's histories as a predict(frame) function."""
    def predict(frame):
        tabular = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
        histories, candidates = encode_pairs(frame, profiles)
        return predict_batches(model, tabular, histories, candidates)
    return predict

# 6. Validation and Test Evaluation

The model has a single fixed configuration, so both splits are reported.

In [ ]:
split_results = []
for split in ("validation", "test"):
    result, _ = score_and_evaluate(
        hybrid_predictor(load_profiles(split)),
        split=split,
        feature_run=RUN,
        output_dir=OUTPUT_DIR,
        model_name=EXPERIMENT_NAME,
        columns=FEATURES,
        results_dir=RESULTS,
    )
    split_results.append(result)

display(pl.concat(split_results).select("model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 7. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(example_user, score_file=OUTPUT_DIR / "validation_scores.parquet", feature_run=RUN, top_n=5)